# Adım 0 — Veri İndirme

Bu notebook, IEEE-CIS Fraud Detection veri setini **Kaggle API** üzerinden otomatik olarak indirir.

**Gereksinim:** Kaggle hesabı ve API token'ı (`kaggle.json`)  
**Veri seti:** [ieee-fraud-detection](https://www.kaggle.com/c/ieee-fraud-detection/data)  
**İndirilen dosyalar:** `train_transaction.csv`, `train_identity.csv`

---

### Kaggle API Token Nasıl Alınır?

1. [kaggle.com](https://www.kaggle.com) → Account → **Create New API Token**
2. İndirilen `kaggle.json` dosyasını aşağıdaki konuma koy:
   - **Linux/Mac:** `~/.kaggle/kaggle.json`
   - **Windows:** `C:\Users\<kullanıcı>\.kaggle\kaggle.json`
3. Dosya izinlerini ayarla (Linux/Mac): `chmod 600 ~/.kaggle/kaggle.json`

## 1. Kaggle Kütüphanesi Kurulumu

In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

try:
    import kaggle
    print("✅ kaggle zaten kurulu.")
except ImportError:
    print("kaggle kuruluyor...")
    install("kaggle")
    import kaggle
    print("✅ kaggle kuruldu.")

## 2. Kaggle API Token Doğrulama

`kaggle.json` dosyasının doğru konumda olup olmadığını kontrol ediyoruz.

In [ ]:
import os
from pathlib import Path

kaggle_dir  = Path.home() / ".kaggle"
kaggle_json = kaggle_dir / "kaggle.json"

if kaggle_json.exists():
    print(f"✅ kaggle.json bulundu: {kaggle_json}")
    # Güvenlik: dosya izinlerini kontrol et
    mode = oct(kaggle_json.stat().st_mode)[-3:]
    if mode != "600":
        print(f"⚠️  Dosya izni {mode} — 600 olmalı.")
        try:
            kaggle_json.chmod(0o600)
            print("   İzin 600 olarak güncellendi.")
        except Exception:
            print("   İzin güncellenemedi (Windows'ta normal).")
    else:
        print("✅ Dosya izni doğru (600).")
else:
    print(f"❌ kaggle.json bulunamadı: {kaggle_json}")
    print()
    print("Yapılması gerekenler:")
    print("  1. kaggle.com → Account → Create New API Token")
    print(f"  2. İndirilen kaggle.json dosyasını buraya koy: {kaggle_json}")
    print("  3. Bu hücreyi tekrar çalıştır.")

## 3. İndirme Dizini Ayarı

Veri dosyalarının indirileceği klasörü belirt.  
Varsayılan olarak notebook ile aynı dizine indirir.

In [ ]:
# İndirme dizini — ihtiyaca göre değiştir
# Örnek: DATA_DIR = Path("../data") veya Path("/kaggle/input")
DATA_DIR = Path(".")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 İndirme dizini: {DATA_DIR.resolve()}")

## 4. Veri Setini İndir

IEEE-CIS Fraud Detection yarışma veri setini indiriyoruz.

> **Not:** Kaggle yarışma veri setlerini indirebilmek için önce  
> [yarışma sayfasında](https://www.kaggle.com/c/ieee-fraud-detection) **Rules** sekmesine girip kabul etmen gerekiyor.

In [ ]:
import zipfile

COMPETITION = "ieee-fraud-detection"

print(f"📥 '{COMPETITION}' indiriliyor...")
print("   (İlk indirmede birkaç dakika sürebilir — ~500MB)")
print()

try:
    from kaggle.api.kaggle_api_extended import KaggleApiExtended
    api = KaggleApiExtended()
    api.authenticate()

    api.competition_download_files(
        competition=COMPETITION,
        path=str(DATA_DIR),
        quiet=False,
    )
    print()
    print("✅ İndirme tamamlandı.")
except Exception as e:
    print(f"❌ Hata: {e}")
    print()
    print("Olası nedenler:")
    print("  - kaggle.json eksik veya hatalı")
    print("  - Yarışma kuralları kabul edilmedi")
    print("    → https://www.kaggle.com/c/ieee-fraud-detection kuralları kabul et")
    print("  - İnternet bağlantısı yok")

## 5. ZIP Dosyasını Çıkar

In [ ]:
zip_path = DATA_DIR / f"{COMPETITION}.zip"

if zip_path.exists():
    print(f"📦 ZIP çıkarılıyor: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print("✅ Çıkarma tamamlandı.")
    print()
    print("Çıkarılan dosyalar:")
    for f in sorted(DATA_DIR.glob("*.csv")):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name:40s} {size_mb:7.1f} MB")
else:
    # Zaten çıkarılmış olabilir
    csvs = list(DATA_DIR.glob("*.csv"))
    if csvs:
        print("ZIP bulunamadı ama CSV dosyaları mevcut:")
        for f in sorted(csvs):
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  {f.name:40s} {size_mb:7.1f} MB")
    else:
        print("❌ ZIP dosyası ve CSV bulunamadı — 4. adımı tekrar çalıştır.")

## 6. Doğrulama

In [ ]:
import pandas as pd

REQUIRED = ["train_transaction.csv", "train_identity.csv"]

print("Dosya doğrulama:")
all_ok = True
for fname in REQUIRED:
    fpath = DATA_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath, nrows=5)
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  ✅ {fname:35s} {size_mb:7.1f} MB  |  kolonlar: {df.shape[1]}")
    else:
        print(f"  ❌ {fname} bulunamadı")
        all_ok = False

print()
if all_ok:
    print("✅ Tüm dosyalar hazır. Adım 1 notebook'una geçebilirsin.")
    print(f"   Veri yolu: {DATA_DIR.resolve()}")
else:
    print("❌ Bazı dosyalar eksik — yukarıdaki adımları kontrol et.")

---

## Notlar

- `test_transaction.csv` ve `test_identity.csv` de indirilir ancak label içermez; bu case'de kullanılmamaktadır
- Veri dosyaları `.gitignore`'a eklenmelidir (büyük dosyalar repoya yüklenmez)
- Adım 1 notebook'unda CSV yolunu bu notebook'daki `DATA_DIR` ile eşleştir